# Skill Domain Classifier using Neural Networks

**Course:** Neural Networks

This notebook trains a small dense neural network that classifies technical skills
into academic/professional domains.

**Architecture:** Input (one-hot) → Dense(16, ReLU) → Dense(5, Softmax) → Output

**Domains:**
- 0: Artificial Intelligence
- 1: Web Development
- 2: Cloud & DevOps
- 3: Data Science
- 4: Databases

In [ ]:
# Install required libraries
# !pip install numpy

In [ ]:
import numpy as np
import json

# Seed for reproducibility
np.random.seed(42)

# Define skills and their domain labels
SKILLS = [
    'Python', 'Java', 'C++', 'C#', 'JavaScript', 'TypeScript',
    'Go', 'Rust', 'PHP', 'Ruby', 'Swift', 'Kotlin', 'Dart', 'R', 'MATLAB',
    'React', 'Angular', 'Vue.js', 'Next.js', 'Node.js', 'Express.js',
    'Django', 'Flask', 'FastAPI', 'Spring Boot', 'Laravel',
    'TensorFlow', 'PyTorch', 'Keras', 'Scikit-Learn', 'Pandas',
    'NumPy', 'OpenCV', 'Hugging Face', 'NLTK', 'spaCy',
    'SQL', 'MySQL', 'PostgreSQL', 'MongoDB', 'Redis',
    'Firebase', 'Supabase', 'Elasticsearch',
    'AWS', 'Google Cloud', 'Microsoft Azure', 'Docker', 'Kubernetes',
    'Linux', 'Git', 'GitHub Actions',
    'Flutter', 'React Native', 'Android Dev', 'iOS Dev',
    'Unity', 'Unreal Engine',
    'Figma', 'UI/UX Design',
    'Blockchain', 'Solidity', 'IoT', 'Robotics', 'Cybersecurity'
]

DOMAINS = ['AI', 'Web Dev', 'Cloud & DevOps', 'Data Science', 'Databases']

# Ground truth labels (domain index for each skill)
LABELS = [
    0, 1, 0, 1, 1, 1,   # Python(AI), Java(Web), C++(AI), C#(Web), JS(Web), TS(Web)
    2, 2, 1, 1, 1, 1, 1, 0, 0,   # Go(Cloud), Rust(Cloud), PHP-Dart(Web), R/MATLAB(AI)
    1, 1, 1, 1, 1, 1,   # React-Express(Web)
    1, 1, 1, 1, 1,       # Django-Laravel(Web)
    0, 0, 0, 0, 3,       # TF/PyTorch/Keras/Sklearn(AI), Pandas(Data)
    3, 0, 0, 0, 0,       # NumPy(Data), OpenCV-spaCy(AI)
    4, 4, 4, 4, 4,       # SQL-Redis(DB)
    4, 4, 4,             # Firebase-Elasticsearch(DB)
    2, 2, 2, 2, 2,       # AWS-Kubernetes(Cloud)
    2, 2, 2,             # Linux-GitHub Actions(Cloud)
    1, 1, 1, 1,          # Flutter-iOS(Web/Mobile)
    1, 1,                 # Unity/Unreal(Web)
    1, 1,                 # Figma/UI-UX(Web)
    2, 2, 2, 0, 2        # Blockchain/Solidity/IoT(Cloud), Robotics(AI), Cybersec(Cloud)
]

NUM_SKILLS = len(SKILLS)
NUM_DOMAINS = len(DOMAINS)
HIDDEN_SIZE = 16

print(f'Skills: {NUM_SKILLS}, Domains: {NUM_DOMAINS}')
print(f'Labels length: {len(LABELS)}')

In [ ]:
# Create one-hot encoded training data
X_train = np.eye(NUM_SKILLS)  # Each skill is a one-hot vector
y_train = np.zeros((NUM_SKILLS, NUM_DOMAINS))
for i, label in enumerate(LABELS):
    y_train[i][label] = 1.0

print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'\nExample: {SKILLS[0]} -> {DOMAINS[LABELS[0]]}')
print(f'  X: {X_train[0][:5]}...')
print(f'  y: {y_train[0]}')

In [ ]:
# Neural Network implementation from scratch

def relu(x):
    """ReLU activation function."""
    return np.maximum(0, x)

def relu_derivative(x):
    """Derivative of ReLU."""
    return (x > 0).astype(float)

def softmax(x):
    """Softmax activation function."""
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def cross_entropy_loss(y_pred, y_true):
    """Cross-entropy loss."""
    return -np.mean(np.sum(y_true * np.log(y_pred + 1e-8), axis=1))


# Initialize weights
W1 = np.random.randn(NUM_SKILLS, HIDDEN_SIZE) * 0.5
b1 = np.zeros((1, HIDDEN_SIZE))
W2 = np.random.randn(HIDDEN_SIZE, NUM_DOMAINS) * 0.5
b2 = np.zeros((1, NUM_DOMAINS))

print(f'W1 shape: {W1.shape}')
print(f'b1 shape: {b1.shape}')
print(f'W2 shape: {W2.shape}')
print(f'b2 shape: {b2.shape}')
print(f'Total parameters: {W1.size + b1.size + W2.size + b2.size}')

In [ ]:
# Training loop
learning_rate = 0.1
epochs = 500
losses = []

for epoch in range(epochs):
    # Forward pass
    z1 = X_train @ W1 + b1       # (N, HIDDEN)
    a1 = relu(z1)                  # (N, HIDDEN)
    z2 = a1 @ W2 + b2             # (N, OUTPUT)
    a2 = softmax(z2)               # (N, OUTPUT)
    
    # Compute loss
    loss = cross_entropy_loss(a2, y_train)
    losses.append(loss)
    
    # Backward pass
    dz2 = a2 - y_train            # (N, OUTPUT)
    dW2 = a1.T @ dz2 / NUM_SKILLS
    db2 = np.mean(dz2, axis=0, keepdims=True)
    
    da1 = dz2 @ W2.T
    dz1 = da1 * relu_derivative(z1)
    dW1 = X_train.T @ dz1 / NUM_SKILLS
    db1 = np.mean(dz1, axis=0, keepdims=True)
    
    # Update weights
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    
    if (epoch + 1) % 100 == 0:
        accuracy = np.mean(np.argmax(a2, axis=1) == np.argmax(y_train, axis=1)) * 100
        print(f'Epoch {epoch+1:4d} | Loss: {loss:.4f} | Accuracy: {accuracy:.1f}%')

print(f'\nFinal Loss: {losses[-1]:.4f}')

In [ ]:
# Test the trained model
def predict(skill_index):
    """Predict domain for a single skill."""
    x = X_train[skill_index:skill_index+1]
    z1 = x @ W1 + b1
    a1 = relu(z1)
    z2 = a1 @ W2 + b2
    a2 = softmax(z2)
    return a2[0]

print('Predictions:')
print('-' * 60)
for i, skill in enumerate(SKILLS):
    probs = predict(i)
    predicted = np.argmax(probs)
    confidence = probs[predicted]
    correct = '✓' if predicted == LABELS[i] else '✗'
    print(f'{correct} {skill:20s} -> {DOMAINS[predicted]:15s} ({confidence:.2%})')

In [ ]:
# Export weights as JSON for the frontend
weights_export = {
    'W1': W1.tolist(),
    'b1': b1[0].tolist(),
    'W2': W2.tolist(),
    'b2': b2[0].tolist(),
    'skills': SKILLS,
    'domains': DOMAINS
}

with open('model_weights.json', 'w') as f:
    json.dump(weights_export, f, indent=2)

print('Weights exported to model_weights.json')
print(f'File would be loaded by src/lib/ml/skillClassifier.ts')